In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PAPERS_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/Project/Data/papers"
)

if not PAPERS_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {PAPERS_DIR}"
    )

pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")

for path in pdf_files:
    print(f"- {path.name}")

Found 5 PDF files:
- Attention.pdf
- BERT.pdf
- NMT.pdf
- ODQA.pdf
- RAG.pdf


In [3]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 44.0 MB/s eta 0:00:00


In [4]:
import pymupdf
import pandas as pd

documents = []

for paper_id, pdf_path in enumerate(pdf_files, start=1):

    with pymupdf.open(pdf_path) as pdf:

        metadata = pdf.metadata or {}
        title = metadata.get("title") or pdf_path.stem

        print(f"Processing: {pdf_path.name}")

        for page_index, page in enumerate(pdf):

            text = page.get_text("text").strip()

            if not text:
                continue

            documents.append({
                "paper_id": paper_id,
                "title": title,
                "source": pdf_path.name,
                "page": page_index + 1,
                "text": text
            })

df = pd.DataFrame(documents)

print("\nExtraction complete")
print("Papers:", df["paper_id"].nunique())
print("Pages extracted:", len(df))
print("Total characters:", df["text"].str.len().sum())

display(df.head(3))

Processing: Attention.pdf
Processing: BERT.pdf
Processing: NMT.pdf
Processing: ODQA.pdf
Processing: RAG.pdf

Extraction complete
Papers: 5
Pages extracted: 73
Total characters: 258848


,paper_id,title,source,page,text
0,1,Attention,Attention.pdf,1,"Provided proper attribution is provided, Googl..."
1,1,Attention,Attention.pdf,2,"1\nIntroduction\nRecurrent neural networks, lo..."
2,1,Attention,Attention.pdf,3,Figure 1: The Transformer - model architecture...


In [5]:
import json

CORPUS_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/Project/"
    "Data/research_corpus.jsonl"
)

with open(CORPUS_PATH, "w", encoding="utf-8") as f:
    for record in documents:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("Corpus saved to:", CORPUS_PATH)

Corpus saved to: /content/drive/MyDrive/Colab Notebooks/Project/Data/research_corpus.jsonl


In [6]:

print("Papers:", df["paper_id"].nunique())
print("Pages:", len(df))
display(df[["title", "source", "page", "text"]].head(3))

Papers: 5
Pages: 73


,title,source,page,text
0,Attention,Attention.pdf,1,"Provided proper attribution is provided, Googl..."
1,Attention,Attention.pdf,2,"1\nIntroduction\nRecurrent neural networks, lo..."
2,Attention,Attention.pdf,3,Figure 1: The Transformer - model architecture...


In [7]:
import re
import pandas as pd

CHUNK_SIZE = 180
CHUNK_OVERLAP = 40

def chunk_text(text, chunk_size=180, overlap=40):
    words = re.findall(r"\S+", text)

    if not words:
        return []

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]

        if not chunk_words:
            break

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks


chunks = []

for row in documents:
    page_chunks = chunk_text(
        row["text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk_index, chunk in enumerate(page_chunks):
        chunks.append({
            "paper_id": row["paper_id"],
            "title": row["title"],
            "source": row["source"],
            "page": row["page"],
            "chunk_id": (
                f'{row["paper_id"]}_'
                f'{row["page"]}_'
                f'{chunk_index}'
            ),
            "text": chunk
        })

chunks_df = pd.DataFrame(chunks)

print("Pages:", len(documents))
print("Chunks:", len(chunks_df))
print("Average words per chunk:",
      round(chunks_df["text"].str.split().str.len().mean(), 1))

display(chunks_df.head(3))

Pages: 73
Chunks: 298
Average words per chunk: 163.4


,paper_id,title,source,page,chunk_id,text
0,1,Attention,Attention.pdf,1,1_1_0,"Provided proper attribution is provided, Googl..."
1,1,Attention,Attention.pdf,1,1_1_1,quality while being more parallelizable and re...
2,1,Attention,Attention.pdf,1,1_1_2,of this work. Noam proposed scaled dot-product...


In [8]:
sample = chunks_df.iloc[0]

print("Chunk ID:", sample["chunk_id"])
print("Source:", sample["source"])
print("Page:", sample["page"])
print("-" * 80)
print(sample["text"])

Chunk ID: 1_1_0
Source: Attention.pdf
Page: 1
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network a

In [9]:
import json

CHUNKS_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/Project/"
    "Data/chunks.jsonl"
)

with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for record in chunks:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("Saved chunks to:", CHUNKS_PATH)

Saved chunks to: /content/drive/MyDrive/Colab Notebooks/Project/Data/chunks.jsonl


In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = chunks_df["text"].tolist()

chunk_embeddings = model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Number of chunks:", len(chunk_texts))
print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Number of chunks: 298
Embedding matrix shape: (298, 384)


In [11]:
def retrieve(query, top_k=5):
  query_embedding = model.encode(
      [query],
      convert_to_numpy=True,
      normalize_embeddings=True
  )[0]

  scores = chunk_embeddings @ query_embedding

  top_indices = np.argsort(scores)[::-1][:top_k]

  results = chunks_df.iloc[top_indices].copy()
  results["similarity"] = scores[top_indices]

  return results

In [12]:
query = "Why does the Transformer use attention instead of recurrence?"

results = retrieve(query, top_k=5)

for _, row in results.iterrows():
    print(f'\nSource: {row["source"]}')
    print(f'Page: {row["page"]}')
    print(f'Chunk: {row["chunk_id"]}')
    print(f'Similarity: {row["similarity"]:.4f}')
    print(row["text"][:700])
    print("-" * 80)


Source: Attention.pdf
Page: 5
Chunk: 1_5_1
Similarity: 0.5638
Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as [38, 2, 9]. • The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can
--------------------------------------------------------------------------------

Source: Attention.pdf
Page: 2
Chunk: 1_2_1
Similarity: 0.5313
through factorization tricks [21] and conditional computation [32], while also improving mod

In [13]:
results = retrieve(
    "How does scaled dot-product attention calculate attention weights?",
    top_k=5
)

display(
    results[["source", "page", "chunk_id", "similarity"]]
)

,source,page,chunk_id,similarity
10,Attention.pdf,4,1_4_0,0.656104
11,Attention.pdf,4,1_4_1,0.619635
12,Attention.pdf,4,1_4_2,0.598767
2,Attention.pdf,1,1_1_2,0.535249
14,Attention.pdf,5,1_5_1,0.423789


In [14]:
eval_questions = [
    "Why does the Transformer eliminate recurrence?",
    "How does scaled dot-product attention work?",
    "What is the purpose of multi-head attention?",
    "How does Bahdanau attention calculate alignment scores?",
    "What is the difference between sparse and dense retrieval?",
    "How does Retrieval-Augmented Generation combine retrieval and generation?",
    "How does Dense Passage Retrieval represent questions and passages?",
    "Why are sentence embeddings useful for semantic search?",
    "What is the role of positional encoding in Transformers?",
    "How can a RAG system determine whether a claim is supported by evidence?"
]

In [15]:
for i, query in enumerate(eval_questions, start=1):
    print(f"\n{'=' * 90}")
    print(f"Q{i}: {query}")

    results = retrieve(query, top_k=3)

    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print(
            f"\nRank: {rank} | "
            f"Chunk: {row['chunk_id']} | "
            f"Page: {row['page']} | "
            f"Score: {row['similarity']:.3f}"
        )
        print(row["text"][:350])


Q1: Why does the Transformer eliminate recurrence?

Rank: 1 | Chunk: 1_2_1 | Page: 2 | Score: 0.346
through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains. Attention mechanisms have become an integral part of compelling sequence modeling and transduc- tion models in various tasks, allowing modeling of d

Rank: 2 | Chunk: 1_5_1 | Page: 5 | Score: 0.338
Applications of Attention in our Model The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input s

Rank: 3 | Chunk: 1_3_0 | Page: 3 | Score: 0.297
Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture u

In [16]:
relevant_chunks = {
    "Q1": {"1_2_1", "1_6_2"},
    "Q2": {"1_4_0", "1_4_1", "1_4_2"},
    "Q3": {"1_5_0", "1_4_2"},
    "Q4": {"3_3_3", "3_4_0", "3_12_3"},
    "Q5": {"4_1_2", "4_2_4", "4_2_5"},
    "Q6": {"5_7_3", "5_9_3"},
    "Q7": {"4_2_5"},
    "Q8": {"2_2_0"},
    "Q9": {"1_6_1"},
    "Q10": {"5_7_2"}
}

In [17]:
def evaluate_retrieval(eval_questions, relevant_chunks, k=5):
    hit_rates = []
    reciprocal_ranks = []

    for i, query in enumerate(eval_questions, start=1):
        qid = f"Q{i}"

        if qid not in relevant_chunks:
            continue

        gold = relevant_chunks[qid]

        # Questions with no supporting evidence are evaluated
        # separately for abstention, not retrieval ranking.
        if not gold:
            continue

        results = retrieve(query, top_k=k)
        ranked_ids = results["chunk_id"].tolist()

        ranks = [
            rank
            for rank, chunk_id in enumerate(ranked_ids, start=1)
            if chunk_id in gold
        ]

        hit_rates.append(float(bool(ranks)))
        reciprocal_ranks.append(
            1 / min(ranks) if ranks else 0.0
        )

    return {
        "Evaluated answerable questions": len(hit_rates),
        f"Hit Rate@{k}": (
            sum(hit_rates) / len(hit_rates)
            if hit_rates else None
        ),
        "MRR": (
            sum(reciprocal_ranks) / len(reciprocal_ranks)
            if reciprocal_ranks else None
        )
    }

In [18]:

evaluate_retrieval(
    eval_questions,
    relevant_chunks,
    k=5
)

{'Evaluated answerable questions': 10,
 'Hit Rate@5': 1.0,
 'MRR': 0.8333333333333334}

In [19]:
import google.genai
import google.auth
MODEL_NAME = "gemini-3.8-flash"
print("Gemini SDK imported successfully")
print("google-auth version:", google.auth.__version__)

Gemini SDK imported successfully
google-auth version: 2.49.0


In [20]:
from google import genai

client = genai.Client(api_key=API_KEY)

interaction = client.interactions.create(
    model="gemini-3.8-flash",
    input="Explain how AI works in a few words"
)
print(interaction.output_text)

It learns patterns from massive data to make predictions.


In [21]:
def generate_rag_answer(query, top_k=5):

    # 1. Retrieve evidence
    results = retrieve(query, top_k=top_k)

    # 2. Format passages with traceable source metadata
    evidence_blocks = []

    for _, row in results.iterrows():
        evidence_blocks.append(
            f"[{row['chunk_id']}]\n"
            f"Source: {row['source']}\n"
            f"Page: {row['page']}\n"
            f"Passage: {row['text']}"
        )

    evidence = "\n\n".join(evidence_blocks)

    # 3. Generate an evidence-grounded answer
    prompt = f"""
You are a research assistant answering questions from research papers.

USER QUESTION:
{query}

RETRIEVED EVIDENCE:
{evidence}

INSTRUCTIONS:
1. Answer using only the retrieved evidence.
2. Cite supporting passages using their exact chunk IDs, e.g. [1_4_1].
3. Do not invent facts, citations, or page references.
4. If the evidence is insufficient, explicitly say what cannot be
   established from the provided passages.
5. Distinguish direct evidence from reasonable inference.
6. Write a concise, technically accurate answer.

ANSWER:
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return {
        "question": query,
        "answer": response.text,
        "retrieved_chunks": results
    }

In [22]:
query = "Why does the Transformer eliminate recurrence?"

result = generate_rag_answer(query)

print("QUESTION:")
print(result["question"])

print("\nGENERATED ANSWER:")
print(result["answer"])

print("\nRETRIEVED SOURCES:")
display(
    result["retrieved_chunks"][
        ["chunk_id", "source", "page", "similarity"]
    ]
)

QUESTION:
Why does the Transformer eliminate recurrence?

GENERATED ANSWER:
Based on the retrieved evidence, the Transformer eliminates recurrence to overcome the fundamental constraint of sequential computation inherent in recurrent models and to achieve the following benefits:

* **Significantly more parallelization:** Relying entirely on an attention mechanism rather than sequential computation allows computations to be parallelized across sequence positions [1_2_1].
* **Faster training and improved efficiency:** This increased parallelization enables the Transformer to be trained much faster, reaching a new state of the art in translation quality after as little as twelve hours of training on eight P100 GPUs [1_2_1].
* **Modeling dependencies directly:** The model can draw global dependencies between input and output purely through the attention mechanism without needing sequence-aligned recurrent networks [1_2_1, 1_2_3].

RETRIEVED SOURCES:


,chunk_id,source,page,similarity
4,1_2_1,Attention.pdf,2,0.345999
14,1_5_1,Attention.pdf,5,0.338284
8,1_3_0,Attention.pdf,3,0.296545
6,1_2_3,Attention.pdf,2,0.283676
7,1_2_4,Attention.pdf,2,0.267087


In [23]:
baseline_result = {
    "question": result["question"],
    "answer": result["answer"],
    "retrieved_chunk_ids": (
        result["retrieved_chunks"]["chunk_id"].tolist()
    )
}

print("Baseline answer saved in memory.")

Baseline answer saved in memory.


In [24]:
import json
import re

def verify_claims(answer, retrieved_chunks):

    evidence = []

    for _, row in retrieved_chunks.iterrows():
        evidence.append({
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "page": int(row["page"]),
            "text": row["text"]
        })

    prompt = f"""
You are a strict evidence verifier for a research-paper RAG system.

Your task is to evaluate factual claims in an answer using ONLY
the supplied evidence.

ANSWER:
{answer}

EVIDENCE:
{json.dumps(evidence, ensure_ascii=False)}

Instructions:
1. Split the answer into atomic factual claims.
2. Evaluate each claim independently.
3. Use exactly one verdict:
   - supported
   - contradicted
   - insufficient_evidence
4. A claim is supported only if the supplied passages entail it.
5. A claim is contradicted only if the passages conflict with it.
6. Use insufficient_evidence when neither support nor contradiction
   can be established.
7. Do not use your general knowledge to fill evidence gaps.
8. Return only valid JSON, with this structure:

{{
  "claims": [
    {{
      "claim": "An atomic factual claim",
      "verdict": "supported",
      "evidence_chunk_ids": ["1_2_1"],
      "reason": "Brief explanation based on the passage"
    }}
  ]
}}

For insufficient_evidence, use an empty evidence_chunk_ids list
unless a passage is relevant to explaining the evidence gap.
"""

    response = client.models.generate_content(
        model= MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    return json.loads(response.text)

In [25]:
verification = verify_claims(
    answer=result["answer"],
    retrieved_chunks=result["retrieved_chunks"]
)

for claim in verification["claims"]:
    print("\nClaim:", claim["claim"])
    print("Verdict:", claim["verdict"])
    print("Evidence:", claim["evidence_chunk_ids"])
    print("Reason:", claim["reason"])


Claim: The Transformer eschews recurrence to address the fundamental constraint of sequential computation.
Verdict: supported
Evidence: ['1_2_1']
Reason: Chunk 1_2_1 notes that 'The fundamental constraint of sequential computation, however, remains' in models using recurrence, and introduces the Transformer as 'a model architecture eschewing recurrence and instead relying entirely on an attention mechanism'.

Claim: The Transformer allows for significantly more parallelization.
Verdict: supported
Evidence: ['1_2_1']
Reason: Chunk 1_2_1 explicitly states: 'The Transformer allows for significantly more parallelization'.

Claim: The Transformer allows computations to be parallelized across sequence positions.
Verdict: insufficient_evidence
Evidence: ['1_2_1']
Reason: Chunk 1_2_1 states that the Transformer allows for significantly more parallelization, but it does not specify that computations are parallelized across sequence positions.

Claim: The increased parallelization enables the T

In [26]:
row = chunks_df.loc[
    chunks_df["chunk_id"] == "1_2_1"
].iloc[0]

print(row["text"])

through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains. Attention mechanisms have become an integral part of compelling sequence modeling and transduc- tion models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network. In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs. 2 Background The goal of reducing sequential computation also for

In [27]:
def check_citation_ids(verification, retrieved_chunks):
    available_ids = set(
        retrieved_chunks["chunk_id"].astype(str)
    )

    for claim in verification["claims"]:
        cited_ids = claim.get("evidence_chunk_ids", [])

        invalid_ids = [
            cid for cid in cited_ids
            if cid not in available_ids
        ]

        claim["citation_ids_valid"] = not invalid_ids
        claim["invalid_citation_ids"] = invalid_ids

    return verification


verification = check_citation_ids(
    verification,
    result["retrieved_chunks"]
)

for claim in verification["claims"]:
    print(
        claim["verdict"],
        "| IDs valid:", claim["citation_ids_valid"],
        "| Claim:", claim["claim"]
    )

supported | IDs valid: True | Claim: The Transformer eschews recurrence to address the fundamental constraint of sequential computation.
supported | IDs valid: True | Claim: The Transformer allows for significantly more parallelization.
insufficient_evidence | IDs valid: True | Claim: The Transformer allows computations to be parallelized across sequence positions.
insufficient_evidence | IDs valid: True | Claim: The increased parallelization enables the Transformer to be trained much faster and with improved efficiency.
supported | IDs valid: True | Claim: The Transformer can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
supported | IDs valid: True | Claim: The Transformer can draw global dependencies between input and output purely through the attention mechanism without needing sequence-aligned recurrent networks.


In [28]:
def repair_claim(claim, top_k=5):
    # Search specifically for evidence supporting this claim
    results = retrieve(claim, top_k=top_k)

    evidence = [
        {
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "page": int(row["page"]),
            "text": row["text"]
        }
        for _, row in results.iterrows()
    ]

    prompt = f"""
You are repairing one factual claim in a research-paper answer.

ORIGINAL CLAIM:
{claim}

NEWLY RETRIEVED EVIDENCE:
{json.dumps(evidence, ensure_ascii=False)}

Rules:
- Use only the supplied evidence.
- If evidence supports the claim, return a concise revised claim.
- If evidence contradicts the claim, correct it using the evidence.
- If evidence is insufficient, do not invent a correction.
- Return JSON only, using this structure:

{{
  "revised_claim": "The revised claim or an empty string",
  "status": "repaired or unresolved",
  "evidence_chunk_ids": ["chunk_id"]
}}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={"response_mime_type": "application/json"}
    )

    repair = json.loads(response.text)
    repair["retrieved_chunks"] = results

    return repair


In [29]:
repair_results = []

for claim_data in verification["claims"]:
    if claim_data["verdict"] == "supported":
        continue

    repair = repair_claim(claim_data["claim"])

    repair_results.append({
        "original_claim": claim_data["claim"],
        "original_verdict": claim_data["verdict"],
        "revised_claim": repair.get("revised_claim", ""),
        "repair_status": repair.get("status", "unresolved"),
        "evidence_chunk_ids": repair.get("evidence_chunk_ids", [])
    })

pd.DataFrame(repair_results)

,original_claim,original_verdict,revised_claim,repair_status,evidence_chunk_ids
0,The Transformer allows computations to be para...,insufficient_evidence,By eschewing recurrence and relying entirely o...,repaired,[1_2_1]
1,The increased parallelization enables the Tran...,insufficient_evidence,The Transformer allows for significantly more ...,repaired,"[1_2_1, 1_8_0]"


In [30]:
test_claim = (
    "The Transformer uses a genetic algorithm "
    "to optimize its attention weights."
)

test_verification = verify_claims(
    answer=test_claim,
    retrieved_chunks=retrieve(test_claim, top_k=5)
)

for claim in test_verification["claims"]:
    print("Claim:", claim["claim"])
    print("Verdict:", claim["verdict"])
    print("Evidence:", claim["evidence_chunk_ids"])
    print("Reason:", claim["reason"])

Claim: The Transformer uses a genetic algorithm to optimize its attention weights.
Verdict: insufficient_evidence
Evidence: []
Reason: The provided passages do not mention genetic algorithms or the optimization technique used for the attention weights.


In [31]:
repair_results = []

for claim_data in test_verification["claims"]:
    if claim_data["verdict"] == "supported":
        continue

    repair = repair_claim(claim_data["claim"])

    repair_results.append({
        "original_claim": claim_data["claim"],
        "original_verdict": claim_data["verdict"],
        "revised_claim": repair.get("revised_claim", ""),
        "repair_status": repair.get("status", "unresolved"),
        "evidence_chunk_ids": repair.get(
            "evidence_chunk_ids", []
        )
    })

display(pd.DataFrame(repair_results))

,original_claim,original_verdict,revised_claim,repair_status,evidence_chunk_ids
0,The Transformer uses a genetic algorithm to op...,insufficient_evidence,,unresolved,[]


In [32]:
def verify_repaired_claim(repair, top_k=5):
    revised_claim = repair.get("revised_claim", "").strip()

    # Nothing to verify if the claim remains unresolved
    if not revised_claim:
        return {
            "revised_claim": "",
            "final_verdict": "unresolved",
            "evidence_chunk_ids": []
        }

    # Verify the revision against the evidence retrieved during repair
    verification = verify_claims(
        answer=revised_claim,
        retrieved_chunks=repair["retrieved_chunks"]
    )

    claims = verification.get("claims", [])

    # Conservative handling: accept only if every returned claim
    # is supported by the retrieved evidence.
    all_supported = (
        len(claims) > 0
        and all(c.get("verdict") == "supported" for c in claims)
    )

    evidence_ids = list(dict.fromkeys(
        cid
        for claim in claims
        if claim.get("verdict") == "supported"
        for cid in claim.get("evidence_chunk_ids", [])
    ))

    return {
        "revised_claim": revised_claim if all_supported else "",
        "final_verdict": "supported" if all_supported else "unresolved",
        "evidence_chunk_ids": evidence_ids
    }

In [34]:
repair = repair_claim(
    "The Transformer uses a genetic algorithm to optimize its attention weights."
)

final_check = verify_repaired_claim(repair)
print(final_check)

{'revised_claim': '', 'final_verdict': 'unresolved', 'evidence_chunk_ids': []}


In [35]:
test_claim = (
    "In scaled dot-product attention, the query-key dot products "
    "are divided by the square root of the key dimension."
)

test_results = retrieve(test_claim, top_k=5)

for _, row in test_results.iterrows():
    print(f"\nChunk: {row['chunk_id']} | Page: {row['page']}")
    print(row["text"][:700])


Chunk: 1_4_2 | Page: 4
counteract this effect, we scale the dot products by 1 √dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and

Chunk: 1_4_0 | Page: 4
Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is computed 

In [36]:
test_verification = verify_claims(
    answer=test_claim,
    retrieved_chunks=test_results
)

print(json.dumps(test_verification, indent=2))

{
  "claims": [
    {
      "claim": "In scaled dot-product attention, the query-key dot products are divided by the square root of the key dimension.",
      "verdict": "supported",
      "evidence_chunk_ids": [
        "1_4_0"
      ],
      "reason": "Chunk 1_4_0 states that keys have dimension dk and that they 'compute the dot products of the query with all keys, divide each by \u221adk'."
    }
  ]
}


In [37]:
test_claim = (
    "In scaled dot-product attention, the query-key dot products "
    "are divided by the square root of the key dimension."
)

test_results = retrieve(test_claim, top_k=5)

for _, row in test_results.iterrows():
    print(f"\nChunk: {row['chunk_id']} | Page: {row['page']}")
    print(f"Similarity: {row['similarity']:.4f}")
    print(row["text"][:700])
    print("-" * 80)


Chunk: 1_4_2 | Page: 4
Similarity: 0.7165
counteract this effect, we scale the dot products by 1 √dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and
--------------------------------------------------------------------------------

Chunk: 1_4_0 | Page: 4
Similarity: 0.6530
Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consis

In [38]:
test_verification = verify_claims(
    answer=test_claim,
    retrieved_chunks=test_results
)

print(json.dumps(test_verification, indent=2))

{
  "claims": [
    {
      "claim": "In scaled dot-product attention, the query-key dot products are divided by the square root of the key dimension.",
      "verdict": "supported",
      "evidence_chunk_ids": [
        "1_4_0"
      ],
      "reason": "Chunk 1_4_0 explains that in Scaled Dot-Product Attention, the keys have dimension dk, and the algorithm computes the dot products of the query with all keys and divides each by \u221adk."
    }
  ]
}


In [39]:
repairable_claim = (
    "In scaled dot-product attention, the query-key dot products "
    "are multiplied by the square root of the key dimension."
)

repairable_results = retrieve(repairable_claim, top_k=5)

for _, row in repairable_results.iterrows():
    print(f"\nChunk: {row['chunk_id']} | Page: {row['page']}")
    print(f"Similarity: {row['similarity']:.4f}")
    print(row["text"][:700])
    print("-" * 80)


Chunk: 1_4_2 | Page: 4
Similarity: 0.7157
counteract this effect, we scale the dot products by 1 √dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To illustrate why the dot products get large, assume that the components of q and k are independent random variables with mean 0 and variance 1. Then their dot product, q · k = Pdk i=1 qiki, has mean 0 and
--------------------------------------------------------------------------------

Chunk: 1_4_0 | Page: 4
Similarity: 0.6487
Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consis

In [40]:
repairable_verification = verify_claims(
    answer=repairable_claim,
    retrieved_chunks=repairable_results
)

print(json.dumps(repairable_verification, indent=2))

{
  "claims": [
    {
      "claim": "In scaled dot-product attention, the query-key dot products are multiplied by the square root of the key dimension.",
      "verdict": "contradicted",
      "evidence_chunk_ids": [
        "1_4_0",
        "1_4_1",
        "1_4_2"
      ],
      "reason": "The passages state that the dot products are divided by \u221adk (or scaled by 1/\u221adk), not multiplied by the square root of the key dimension."
    }
  ]
}


In [41]:
repair = repair_claim(
    claim=repairable_claim,
    top_k=5
)

print(json.dumps(
    {
        "original_claim": repairable_claim,
        "revised_claim": repair.get("revised_claim", ""),
        "repair_status": repair.get("status", "unresolved"),
        "evidence_chunk_ids": repair.get("evidence_chunk_ids", [])
    },
    indent=2
))

{
  "original_claim": "In scaled dot-product attention, the query-key dot products are multiplied by the square root of the key dimension.",
  "revised_claim": "In scaled dot-product attention, the query-key dot products are divided by the square root of the key dimension.",
  "repair_status": "repaired",
  "evidence_chunk_ids": [
    "1_4_0",
    "1_4_1"
  ]
}


In [43]:
final_check = verify_repaired_claim(repair)
print(json.dumps(final_check, indent=2))

{
  "revised_claim": "In scaled dot-product attention, the query-key dot products are divided by the square root of the key dimension.",
  "final_verdict": "supported",
  "evidence_chunk_ids": [
    "1_4_0"
  ]
}


In [44]:
import pandas as pd

eval_data = [
    # Directly answerable
    {"id": 1, "question": "Why does the Transformer eliminate recurrence?",
     "category": "answerable"},
    {"id": 2, "question": "How does scaled dot-product attention work?",
     "category": "answerable"},
    {"id": 3, "question": "What is the purpose of multi-head attention?",
     "category": "answerable"},
    {"id": 4, "question": "How does Bahdanau attention calculate alignment scores?",
     "category": "answerable"},
    {"id": 5, "question": "What is the difference between sparse and dense retrieval?",
     "category": "answerable"},
    {"id": 6, "question": "How does RAG combine retrieval and generation?",
     "category": "answerable"},
    {"id": 7, "question": "How does DPR represent questions and passages?",
     "category": "answerable"},
    {"id": 8, "question": "Why are sentence embeddings useful for semantic search?",
     "category": "answerable"},
    {"id": 9, "question": "What is the role of positional encoding?",
     "category": "answerable"},
    {"id": 10, "question": "How does RAG use retrieved passages to generate answers?",
     "category": "answerable"},
    {"id": 11, "question": "What is additive attention?",
     "category": "answerable"},
    {"id": 12, "question": "How is dot-product attention computationally efficient?",
     "category": "answerable"},
    {"id": 13, "question": "What is the purpose of the encoder-decoder architecture in neural machine translation?",
     "category": "answerable"},
    {"id": 14, "question": "What is the role of the softmax function in scaled dot-product attention?",
     "category": "answerable"},
    {"id": 15, "question": "How are query, key, and value vectors used in attention?",
     "category": "answerable"},

    # Incorrect claims that should be repairable
    {"id": 16, "question": "In scaled dot-product attention, are query-key dot products multiplied by the square root of the key dimension?",
     "category": "repairable"},
    {"id": 17, "question": "Does the Transformer rely on recurrence to process sequences?",
     "category": "repairable"},
    {"id": 18, "question": "Does multi-head attention use only one learned projection of queries, keys, and values?",
     "category": "repairable"},
    {"id": 19, "question": "Does DPR encode questions and passages using exactly the same encoder parameters?",
     "category": "repairable"},
    {"id": 20, "question": "Does Sentence-BERT require comparing every pair of sentences with a cross-encoder at search time?",
     "category": "repairable"},
    {"id": 21, "question": "Does RAG generate answers without conditioning on retrieved documents?",
     "category": "repairable"},
    {"id": 22, "question": "Does positional encoding provide the Transformer with no information about token positions?",
     "category": "repairable"},
    {"id": 23, "question": "Does additive attention compute compatibility using a dot product alone?",
     "category": "repairable"},

    # Claims/questions that may be unanswerable from this corpus
    {"id": 24, "question": "What exact GPU temperature did the authors observe during Transformer training?",
     "category": "insufficient_evidence"},
    {"id": 25, "question": "What was the electricity cost of training the original RAG model?",
     "category": "insufficient_evidence"},
    {"id": 26, "question": "What exact salary did each author receive while conducting the research?",
     "category": "insufficient_evidence"},
    {"id": 27, "question": "What was the authors' private opinion of every competing retrieval system?",
     "category": "insufficient_evidence"},
    {"id": 28, "question": "What was the precise network latency of every experiment in the DPR paper?",
     "category": "insufficient_evidence"},
    {"id": 29, "question": "What exact hardware temperature was recorded during Sentence-BERT experiments?",
     "category": "insufficient_evidence"},
    {"id": 30, "question": "What private discussions led the authors to select their final model architectures?",
     "category": "insufficient_evidence"},
]

eval_df = pd.DataFrame(eval_data)
display(eval_df)

,id,question,category
0,1,Why does the Transformer eliminate recurrence?,answerable
1,2,How does scaled dot-product attention work?,answerable
2,3,What is the purpose of multi-head attention?,answerable
3,4,How does Bahdanau attention calculate alignmen...,answerable
4,5,What is the difference between sparse and dens...,answerable
5,6,How does RAG combine retrieval and generation?,answerable
6,7,How does DPR represent questions and passages?,answerable
7,8,Why are sentence embeddings useful for semanti...,answerable
8,9,What is the role of positional encoding?,answerable
9,10,How does RAG use retrieved passages to generat...,answerable


In [45]:
retrieval_records = []

for item in eval_data:
    results = retrieve(item["question"], top_k=5)

    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        retrieval_records.append({
            "question_id": item["id"],
            "question": item["question"],
            "category": item["category"],
            "rank": rank,
            "chunk_id": row["chunk_id"],
            "source": row["source"],
            "page": int(row["page"]),
            "similarity": float(row["similarity"]),
            "text": row["text"]
        })

retrieval_df = pd.DataFrame(retrieval_records)

print("Questions:", len(eval_df))
print("Retrieved passages:", len(retrieval_df))

display(retrieval_df.head(10))

Questions: 30
Retrieved passages: 150


,question_id,question,category,rank,chunk_id,source,page,similarity,text
0,1,Why does the Transformer eliminate recurrence?,answerable,1,1_2_1,Attention.pdf,2,0.345999,through factorization tricks [21] and conditio...
1,1,Why does the Transformer eliminate recurrence?,answerable,2,1_5_1,Attention.pdf,5,0.338284,Applications of Attention in our Model The Tra...
2,1,Why does the Transformer eliminate recurrence?,answerable,3,1_3_0,Attention.pdf,3,0.296545,Figure 1: The Transformer - model architecture...
3,1,Why does the Transformer eliminate recurrence?,answerable,4,1_2_3,Attention.pdf,2,0.283676,in order to compute a representation of the se...
4,1,Why does the Transformer eliminate recurrence?,answerable,5,1_2_4,Attention.pdf,2,0.267087,encoder maps an input sequence of symbol repre...
5,2,How does scaled dot-product attention work?,answerable,1,1_4_2,Attention.pdf,4,0.666730,"counteract this effect, we scale the dot produ..."
6,2,How does scaled dot-product attention work?,answerable,2,1_4_1,Attention.pdf,4,0.642469,"as: Attention(Q, K, V ) = softmax(QKT √dk )V (..."
7,2,How does scaled dot-product attention work?,answerable,3,1_4_0,Attention.pdf,4,0.617157,Scaled Dot-Product Attention Multi-Head Attent...
8,2,How does scaled dot-product attention work?,answerable,4,1_1_2,Attention.pdf,1,0.554508,of this work. Noam proposed scaled dot-product...
9,2,How does scaled dot-product attention work?,answerable,5,1_5_1,Attention.pdf,5,0.447338,Applications of Attention in our Model The Tra...


In [46]:
output_dir = Path("/content/drive/MyDrive/Colab Notebooks/Project/Data")
output_dir.mkdir(parents=True, exist_ok=True)

eval_df.to_csv(
    output_dir / "eval_questions.csv",
    index=False
)

retrieval_df.to_json(
    output_dir / "eval_retrieval_candidates.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved evaluation questions and retrieval candidates.")

Saved evaluation questions and retrieval candidates.


In [47]:
# Display the five retrieved passages for one question at a time.

def inspect_question(question_id):
    rows = retrieval_df[
        retrieval_df["question_id"] == question_id
    ].sort_values("rank")

    if rows.empty:
        print("Question ID not found.")
        return

    print("Question:", rows.iloc[0]["question"])
    print("Category:", rows.iloc[0]["category"])

    for _, row in rows.iterrows():
        print(
            f"\nRank {row['rank']} | "
            f"Chunk: {row['chunk_id']} | "
            f"Page: {row['page']} | "
            f"Similarity: {row['similarity']:.3f}"
        )
        print(row["text"])
        print("-" * 80)

In [48]:
from pathlib import Path

review_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/Project/Data"
    "all_questions_evidence_review.md"
)

with open(review_path, "w", encoding="utf-8") as f:
    f.write("# RAG Evaluation: Evidence Review\n\n")

    for question_id in sorted(retrieval_df["question_id"].unique()):
        rows = (
            retrieval_df[
                retrieval_df["question_id"] == question_id
            ]
            .sort_values("rank")
        )

        first = rows.iloc[0]

        f.write(f"## Question {question_id}\n\n")
        f.write(f"**Question:** {first['question']}\n\n")
        f.write(f"**Original category:** {first['category']}\n\n")

        for _, row in rows.iterrows():
            f.write(
                f"### Rank {row['rank']} — "
                f"{row['chunk_id']} "
                f"(page {row['page']}, "
                f"similarity {row['similarity']:.3f})\n\n"
            )
            f.write(f"**Source:** {row['source']}\n\n")
            f.write(row["text"].strip() + "\n\n")
            f.write("---\n\n")

print(f"Saved review file: {review_path}")
print(f"Questions: {retrieval_df['question_id'].nunique()}")
print(f"Passages: {len(retrieval_df)}")

Saved review file: /content/drive/MyDrive/Colab Notebooks/Project/Dataall_questions_evidence_review.md
Questions: 30
Passages: 150


In [49]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Colab Notebooks/"
    "Project/Dataall_questions_evidence_review.md"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [63]:
!pip -q install -U sentence-transformers

from sentence_transformers import CrossEncoder

nli_model = CrossEncoder(
    "cross-encoder/nli-deberta-v3-xsmall",
    device="cpu"
)

print("Local NLI model loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 14.9 MB/s eta 0:00:00


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  283MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Local NLI model loaded.


In [65]:
import numpy as np

NLI_LABELS = ["contradiction", "entailment", "neutral"]

def local_verify_claim(claim, retrieved_chunks):
    pairs = [
        (str(row["text"]), claim)
        for _, row in retrieved_chunks.iterrows()
    ]

    scores = nli_model.predict(pairs)

    best = None

    for i, (_, row) in enumerate(retrieved_chunks.iterrows()):
        logits = scores[i]
        probabilities = np.exp(logits - np.max(logits))
        probabilities = probabilities / probabilities.sum()

        result = {
            "chunk_id": str(row["chunk_id"]),
            "text": row["text"],
            "scores": dict(zip(
                NLI_LABELS, probabilities.astype(float)
            )),
            "predicted_label": NLI_LABELS[int(np.argmax(logits))]
        }

        # Find the passage with the strongest entailment score.
        if best is None or (
            result["scores"]["entailment"]
            > best["scores"]["entailment"]
        ):
            best = result

    if best is None:
        return {
            "claim": claim,
            "verdict": "insufficient_evidence",
            "evidence_chunk_id": None,
            "scores": None
        }

    verdict_map = {
        "entailment": "supported",
        "contradiction": "contradicted",
        "neutral": "insufficient_evidence"
    }

    return {
        "claim": claim,
        "verdict": verdict_map[best["predicted_label"]],
        "evidence_chunk_id": best["chunk_id"],
        "scores": best["scores"],
        "evidence_text": best["text"]
    }

In [66]:
test_claim = (
    "The Transformer uses a genetic algorithm "
    "to optimize its attention weights."
)

test_chunks = retrieve(test_claim, top_k=5)

local_result = local_verify_claim(test_claim, test_chunks)

print("Verdict:", local_result["verdict"])
print("Evidence chunk:", local_result["evidence_chunk_id"])
print("Scores:", local_result["scores"])
print("Evidence:", local_result["evidence_text"][:500])

Verdict: insufficient_evidence
Evidence chunk: 1_5_0
Scores: {'contradiction': np.float64(0.2692802846431732), 'entailment': np.float64(0.022103849798440933), 'neutral': np.float64(0.7086158394813538)}
Evidence: output values. These are concatenated and once again projected, resulting in the final values, as depicted in Figure 2. Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. MultiHead(Q, K, V ) = Concat(head1, ..., headh)W O where headi = Attention(QW Q i , KW K i , V W V i ) Where the projections are parameter matrices W Q i ∈Rdmodel×dk, W K i ∈Rdmodel×dk, W V 


In [67]:
def local_verify_claim(claim, retrieved_chunks):
    if retrieved_chunks.empty:
        return {
            "claim": claim,
            "verdict": "insufficient_evidence",
            "evidence_chunk_id": None
        }

    pairs = [
        (str(row["text"]), claim)
        for _, row in retrieved_chunks.iterrows()
    ]

    logits = nli_model.predict(pairs)

    # Convert logits to probabilities for each passage.
    probs = []
    for row_logits in logits:
        p = np.exp(row_logits - np.max(row_logits))
        probs.append(p / p.sum())

    probs = np.asarray(probs)

    # Class order: contradiction, entailment, neutral.
    best_contradiction = int(np.argmax(probs[:, 0]))
    best_entailment = int(np.argmax(probs[:, 1]))

    max_contradiction = float(probs[:, 0].max())
    max_entailment = float(probs[:, 1].max())

    # Conservative thresholds; validate these on a manually
    # labelled sample before using them in reported results.
    threshold = 0.70

    if max_contradiction >= threshold:
        idx = best_contradiction
        verdict = "contradicted"
    elif max_entailment >= threshold:
        idx = best_entailment
        verdict = "supported"
    else:
        idx = int(np.argmax(probs[:, 2]))
        verdict = "insufficient_evidence"

    return {
        "claim": claim,
        "verdict": verdict,
        "evidence_chunk_id": str(
            retrieved_chunks.iloc[idx]["chunk_id"]
        ),
        "scores": {
            "contradiction": float(probs[idx, 0]),
            "entailment": float(probs[idx, 1]),
            "neutral": float(probs[idx, 2])
        },
        "evidence_text": str(retrieved_chunks.iloc[idx]["text"])
    }

In [68]:
test_chunks = retrieve(
    "The Transformer uses a genetic algorithm to optimize its attention weights.",
    top_k=5
)

result = local_verify_claim(test_claim, test_chunks)

print("Verdict:", result["verdict"])
print("Evidence chunk:", result["evidence_chunk_id"])
print("Scores:", result["scores"])

Verdict: contradicted
Evidence chunk: 1_2_1
Scores: {'contradiction': 0.8646996021270752, 'entailment': 0.002131580375134945, 'neutral': 0.13316881656646729}


In [69]:
import pandas as pd

repairable = [
    item for item in eval_data
    if item["category"] == "repairable"
]

rows = []

for item in repairable:
    claim = item["question"]
    chunks = retrieve(claim, top_k=5)
    result = local_verify_claim(claim, chunks)

    rows.append({
        "question_id": item["id"],
        "claim": claim,
        "predicted_verdict": result["verdict"],
        "evidence_chunk_id": result["evidence_chunk_id"],
        "scores": result.get("scores")
    })

local_results_df = pd.DataFrame(rows)
display(local_results_df)

,question_id,claim,predicted_verdict,evidence_chunk_id,scores
0,16,"In scaled dot-product attention, are query-key...",contradicted,1_4_2,"{'contradiction': 0.7060143351554871, 'entailm..."
1,17,Does the Transformer rely on recurrence to pro...,contradicted,1_1_0,"{'contradiction': 0.7951224446296692, 'entailm..."
2,18,Does multi-head attention use only one learned...,contradicted,1_4_2,"{'contradiction': 0.9868631362915039, 'entailm..."
3,19,Does DPR encode questions and passages using e...,contradicted,4_9_2,"{'contradiction': 0.8666514754295349, 'entailm..."
4,20,Does Sentence-BERT require comparing every pai...,contradicted,2_8_4,"{'contradiction': 0.753334105014801, 'entailme..."
5,21,Does RAG generate answers without conditioning...,insufficient_evidence,5_8_2,"{'contradiction': 0.03275170177221298, 'entail..."
6,22,Does positional encoding provide the Transform...,contradicted,1_9_0,"{'contradiction': 0.9742032289505005, 'entailm..."
7,23,Does additive attention compute compatibility ...,contradicted,1_4_2,"{'contradiction': 0.8063765168190002, 'entailm..."


In [70]:
claim_map = {
    16: "In scaled dot-product attention, query-key dot products are multiplied by the square root of the key dimension.",
    17: "The Transformer relies on recurrence to process sequences.",
    18: "Multi-head attention uses only one learned projection of queries, keys, and values.",
    19: "DPR encodes questions and passages using exactly the same encoder parameters.",
    20: "Sentence-BERT requires comparing every pair of sentences with a cross-encoder at search time.",
    21: "RAG generates answers without conditioning on retrieved documents.",
    22: "Positional encoding provides the Transformer with no information about token positions.",
    23: "Additive attention computes compatibility using a dot product alone."
}

ground_truth = {qid: "contradicted" for qid in claim_map}

rows = []

for qid, claim in claim_map.items():
    chunks = retrieve(claim, top_k=5)
    result = local_verify_claim(claim, chunks)

    rows.append({
        "question_id": qid,
        "claim": claim,
        "ground_truth": ground_truth[qid],
        "prediction": result["verdict"],
        "correct": result["verdict"] == ground_truth[qid],
        "evidence_chunk_id": result["evidence_chunk_id"]
    })

claim_eval_df = pd.DataFrame(rows)

display(claim_eval_df)
print("Accuracy:", round(claim_eval_df["correct"].mean(), 3))

,question_id,claim,ground_truth,prediction,correct,evidence_chunk_id
0,16,"In scaled dot-product attention, query-key dot...",contradicted,insufficient_evidence,False,1_4_1
1,17,The Transformer relies on recurrence to proces...,contradicted,contradicted,True,1_2_1
2,18,Multi-head attention uses only one learned pro...,contradicted,contradicted,True,1_4_2
3,19,DPR encodes questions and passages using exact...,contradicted,contradicted,True,4_3_0
4,20,Sentence-BERT requires comparing every pair of...,contradicted,insufficient_evidence,False,2_2_3
5,21,RAG generates answers without conditioning on ...,contradicted,contradicted,True,5_4_4
6,22,Positional encoding provides the Transformer w...,contradicted,contradicted,True,1_9_0
7,23,Additive attention computes compatibility usin...,contradicted,contradicted,True,1_4_2


Accuracy: 0.75


In [72]:

claim_eval_df.to_csv(
    OUT_DIR / "local_claim_verification.csv",
    index=False
)

In [74]:
failure_claims = {
    16: claim_map[16],
    20: claim_map[20],
}

for qid, claim in failure_claims.items():
    print(f"\n{'=' * 70}\nQ{qid}: {claim}\n")

    chunks = retrieve(claim, top_k=5)
    pairs = [(str(row["text"]), claim) for _, row in chunks.iterrows()]
    logits = nli_model.predict(pairs)

    for i, (_, row) in enumerate(chunks.iterrows()):
        p = np.exp(logits[i] - np.max(logits[i]))
        p = p / p.sum()

        print(
            f"\nChunk: {row['chunk_id']} | "
            f"contradiction={p[0]:.3f}, "
            f"entailment={p[1]:.3f}, "
            f"neutral={p[2]:.3f}"
        )
        print(str(row["text"])[:500])


Q16: In scaled dot-product attention, query-key dot products are multiplied by the square root of the key dimension.


Chunk: 1_4_2 | contradiction=0.267, entailment=0.008, neutral=0.724
counteract this effect, we scale the dot products by 1 √dk . 3.2.2 Multi-Head Attention Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional 4To il

Chunk: 1_4_0 | contradiction=0.201, entailment=0.006, neutral=0.793
Scaled Dot-Product Attention Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several attention layers running in parallel. of the values, where the weight assigned to each value is 

In [75]:
corrected_claims = {
    16: (
        "Scaled dot-product attention divides query-key dot products "
        "by the square root of the key dimension."
    ),
    20: (
        "Sentence-BERT computes sentence embeddings independently, "
        "allowing semantic search using vector similarity instead "
        "of comparing every sentence pair with a cross-encoder."
    ),
}

for qid, corrected in corrected_claims.items():
    print(f"\nQ{qid}: {corrected}")

    chunks = retrieve(corrected, top_k=5)
    result = local_verify_claim(corrected, chunks)

    print("Verdict:", result["verdict"])
    print("Evidence chunk:", result["evidence_chunk_id"])
    print("Scores:", result.get("scores"))
    print("Evidence:", result.get("evidence_text", "")[:400])


Q16: Scaled dot-product attention divides query-key dot products by the square root of the key dimension.
Verdict: insufficient_evidence
Evidence chunk: 1_4_1
Scores: {'contradiction': 0.007031853776425123, 'entailment': 0.0023952254559844732, 'neutral': 0.9905728697776794}
Evidence: as: Attention(Q, K, V ) = softmax(QKT √dk )V (1) The two most commonly used attention functions are additive attention [2], and dot-product (multi- plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of 1 √dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are simila

Q20: Sentence-BERT computes sentence embeddings independently, allowing semantic search using vector similarity instead of comparing every sentence pair with a cross-encoder.
Verdict: insufficient_evidence
Evidence chunk: 2_1_3
Scores: {'contradiction': 0.003949328325688839, 'entailment': 0.0016989981522783637,

In [76]:
abstention_questions = {
    24: "What exact GPU temperature did the authors observe during Transformer training?",
    25: "What was the electricity cost of training the original RAG model?",
    26: "What exact salary did each author receive while conducting the research?",
    27: "What was the authors' private opinion of every competing retrieval system?",
    28: "What was the precise network latency of every experiment in the DPR paper?",
    29: "What exact hardware temperature was recorded during Sentence-BERT experiments?",
    30: "What private discussions led the authors to select their final model architectures?"
}

abstention_rows = []

for qid, question in abstention_questions.items():
    chunks = retrieve(question, top_k=5)

    # Check the question against the retrieved passages.
    result = local_verify_claim(question, chunks)

    abstention_rows.append({
        "question_id": qid,
        "question": question,
        "predicted_verdict": result["verdict"],
        "evidence_chunk_id": result["evidence_chunk_id"]
    })

abstention_df = pd.DataFrame(abstention_rows)
display(abstention_df)

,question_id,question,predicted_verdict,evidence_chunk_id
0,24,What exact GPU temperature did the authors obs...,contradicted,1_9_0
1,25,What was the electricity cost of training the ...,contradicted,5_8_0
2,26,What exact salary did each author receive whil...,contradicted,5_7_0
3,27,What was the authors' private opinion of every...,contradicted,4_10_3
4,28,What was the precise network latency of every ...,contradicted,3_14_0
5,29,What exact hardware temperature was recorded d...,contradicted,2_4_0
6,30,What private discussions led the authors to se...,contradicted,1_9_0


In [78]:
print("Label mapping:", nli_model.model.config.id2label)
print("Reverse mapping:", nli_model.model.config.label2id)
print("Number of labels:", nli_model.model.config.num_labels)

Label mapping: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}
Reverse mapping: {'contradiction': 0, 'entailment': 1, 'neutral': 2}
Number of labels: 3


In [81]:
import pandas as pd

abstention_rows = []

for qid, question in abstention_questions.items():
    retrieved = retrieve(question, top_k=5)

    top = retrieved.iloc[0]

    abstention_rows.append({
        "question_id": qid,
        "question": question,
        "top_retrieval_similarity": float(top["similarity"]),
        "top_chunk_id": top["chunk_id"],
        "top_chunk_text": top["text"],
        "expected_verdict": "insufficient_evidence",
        "needs_manual_review": True
    })

abstention_df = pd.DataFrame(abstention_rows)

display(
    abstention_df[
        [
            "question_id",
            "top_retrieval_similarity",
            "top_chunk_id",
            "expected_verdict",
            "needs_manual_review"
        ]
    ]
)

,question_id,top_retrieval_similarity,top_chunk_id,expected_verdict,needs_manual_review
0,24,0.480691,1_8_3,insufficient_evidence,True
1,25,0.332588,5_8_0,insufficient_evidence,True
2,26,0.246022,5_7_0,insufficient_evidence,True
3,27,0.458209,4_4_4,insufficient_evidence,True
4,28,0.394448,4_8_3,insufficient_evidence,True
5,29,0.397497,2_4_0,insufficient_evidence,True
6,30,0.278732,5_14_3,insufficient_evidence,True


In [82]:
# Inspect all five retrieved passages for each unanswerable question.
review_rows = []

for qid, question in abstention_questions.items():
    retrieved = retrieve(question, top_k=5)

    for rank, (_, chunk) in enumerate(retrieved.iterrows(), start=1):
        review_rows.append({
            "question_id": qid,
            "question": question,
            "rank": rank,
            "similarity": float(chunk["similarity"]),
            "chunk_id": chunk["chunk_id"],
            "paper": chunk["title"],
            "evidence_text": chunk["text"]
        })

review_df = pd.DataFrame(review_rows)

display(
    review_df[
        ["question_id", "rank", "similarity", "chunk_id",
         "paper", "evidence_text"]
    ].sort_values(["question_id", "rank"])
)


,question_id,rank,similarity,chunk_id,paper,evidence_text
0,24,1,0.480691,1_8_3,Attention,"to input length + 50, but terminate early when..."
1,24,2,0.373035,1_8_0,Attention,Table 2: The Transformer achieves better BLEU ...
2,24,3,0.355586,1_8_1,Attention,is added to the sub-layer input and normalized...
3,24,4,0.336409,4_7_5,ODQA,"allowed, we ﬁne-tune the reader trained on Nat..."
4,24,5,0.315064,1_9_0,Attention,Table 3: Variations on the Transformer archite...
5,25,1,0.332588,5_8_0,RAG,Table 4: Human assessments for the Jeopardy Qu...
6,25,2,0.331420,5_17_2,RAG,"is sufﬁciently fast on CPU, so we store docume..."
7,25,3,0.303838,5_4_0,RAG,minimize the negative marginal log-likelihood ...
8,25,4,0.287330,1_8_0,Attention,Table 2: The Transformer achieves better BLEU ...
9,25,5,0.283530,5_6_2,RAG,that RAG generations are more diverse than BAR...
